In [ ]:
# 📦 Imports for training & saving embeddings
import os
import cv2
import numpy as np
import pickle
from insightface.app import FaceAnalysis

# 🚀 Initialize InsightFace model
model = FaceAnalysis(name='buffalo_l')
model.prepare(ctx_id=0, det_thresh=0.1, det_size=(160, 160))  # Use -1 for CPU

# 📁 Directory with labeled training images
training_path = '../data/hb-uzb'
known_embeddings = []
classNames = []

os.makedirs('training_images', exist_ok=True)

# 📌 Process each image
for file in os.listdir(training_path):
    img_path = os.path.join(training_path, file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    faces = model.get(img_rgb)

    if faces:
        emb = faces[0].embedding
        x1, y1, x2, y2 = faces[0].bbox.astype(int)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)


        name = os.path.splitext(file)[0]
        # Save the cropped face image
        # if face_crop.size > 0:
        #  cv2.imwrite(f"training_images/cropped_{name}", face_crop)

        known_embeddings.append(emb)
        
        print(f"✅ Detected {name} in {file}")
        classNames.append(name)

        cv2.imshow(f"Detected", img)
        cv2.waitKey(0)

    else:
        print(f"⚠️ No face detected in {file}")

cv2.destroyAllWindows()

# 💾 Save embeddings + names
data = {
    'embeddings': np.array(known_embeddings),
    'names': classNames
}
with open('face_data_hb-uzb.pkl', 'wb') as f:
    pickle.dump(data, f)

print("✅ Saved face embeddings to face_data_hb-uzb.pkl")

### ARCFaceOnnx Experiment